In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from pyspark.dbutils import *
from datetime import *

In [0]:
# DEBUG — print all received widget values
try:
    p_start     = dbutils.widgets.get("p_start_date")
except:
    p_start     = "NOT_FOUND"

try:
    p_end       = dbutils.widgets.get("p_end_date")
except:
    p_end       = "NOT_FOUND"

try:
    p_ingestion = dbutils.widgets.get("p_ingestion_date")
except:
    p_ingestion = "NOT_FOUND"

print(f"p_start_date:     '{p_start}'")
print(f"p_end_date:       '{p_end}'")
print(f"p_ingestion_date: '{p_ingestion}'")

In [0]:
# ============================================================
# BRONZE TO SILVER NOTEBOOK
# Reads raw Open-Meteo JSON from bronze, explodes hourly arrays
# into individual rows, applies cleaning and DQ checks,
# writes cleaned Delta table to silver layer
# ============================================================

# Configuration
STORAGE_ACCOUNT = "weatherdatalake"
CONTAINER       = "weather-data"
CATALOG         = "weather_catalog"
SILVER_SCHEMA   = "silver"
SILVER_TABLE    = "weather_hourly_cleaned"

BRONZE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze/open_meteo/raw/2026/*/*/*/*.json"
SILVER_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver/open_meteo/cleaned/"

from datetime import datetime, timezone

def get_widget(name, default):
    try:
        dbutils.widgets.remove(name)
    except:
        pass
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name)

start_date = get_widget("p_start_date", "2024-01-02")
end_date   = get_widget("p_end_date",   "2024-01-02")

# Always compute today fresh — never rely on widget for this
ingestion_date = datetime.now(timezone.utc).strftime("%Y/%m/%d")

print(f"start_date:     {start_date}")
print(f"end_date:       {end_date}")
print(f"ingestion_date: {ingestion_date}")

start_date:     
end_date:       
ingestion_date: 2026/05/24


In [0]:
from pyspark.sql.functions import col, regexp_extract

# Build path from freshly computed ingestion_date — never from widget
bronze_path = (
    f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}"
    f".dfs.core.windows.net/bronze/open_meteo/raw/{ingestion_date}/"
)

print(f"Reading: {bronze_path}")

# Verify path exists
try:
    city_folders = dbutils.fs.ls(bronze_path)
    print(f"Found {len(city_folders)} folders: {[f.name for f in city_folders]}")
except Exception as e:
    raise Exception(f"Bronze path not found: {bronze_path} | Error: {e}")

# Read all city files
df_raw = spark.read \
    .option("multiline", "true") \
    .option("recursiveFileLookup", "true") \
    .json(bronze_path)

print(f"Raw records: {df_raw.count()}")

Reading: abfss://weather-data@weatherdatalake.dfs.core.windows.net/bronze/open_meteo/raw/2026/05/24/
Found 10 folders: ['Berlin/', 'Dubai/', 'London/', 'Mumbai/', 'New_York/', 'Paris/', 'Singapore/', 'Sydney/', 'Tokyo/', 'Toronto/']
Raw records: 10


In [0]:
from pyspark.sql.functions import arrays_zip, explode

df_with_meta = df_raw.withColumn("source_file", col("_metadata.file_path"))

# Extract city from second-to-last path segment
# .../2026/05/24/Singapore/Singapore_2024-01-02.json → Singapore
df_with_meta = df_with_meta.withColumn(
    "city_name",
    regexp_extract(col("source_file"), r"/([^/]+)/[^/]+\.json$", 1)
)

print("Cities found:")
df_with_meta.select("city_name").distinct().show()

# Zip hourly arrays
df_zipped = df_with_meta.select(
    col("city_name"),
    col("latitude"),
    col("longitude"),
    col("elevation"),
    col("timezone"),
    col("timezone_abbreviation"),
    col("utc_offset_seconds"),
    arrays_zip(
        col("hourly.time"),
        col("hourly.temperature_2m"),
        col("hourly.apparent_temperature"),
        col("hourly.relativehumidity_2m"),
        col("hourly.dewpoint_2m"),
        col("hourly.precipitation"),
        col("hourly.rain"),
        col("hourly.snowfall"),
        col("hourly.weathercode"),
        col("hourly.cloudcover"),
        col("hourly.windspeed_10m"),
        col("hourly.windgusts_10m"),
        col("hourly.winddirection_10m"),
        col("hourly.surface_pressure")
    ).alias("hourly_zipped")
)

df_exploded = df_zipped.withColumn("hourly_data", explode(col("hourly_zipped")))

df_flat = df_exploded.select(
    col("city_name"),
    col("latitude"),
    col("longitude"),
    col("elevation"),
    col("timezone"),
    col("timezone_abbreviation"),
    col("utc_offset_seconds"),
    col("hourly_data.time").alias("observation_time"),
    col("hourly_data.temperature_2m").alias("temperature_c"),
    col("hourly_data.apparent_temperature").alias("apparent_temperature_c"),
    col("hourly_data.relativehumidity_2m").alias("relative_humidity_pct"),
    col("hourly_data.dewpoint_2m").alias("dewpoint_c"),
    col("hourly_data.precipitation").alias("precipitation_mm"),
    col("hourly_data.rain").alias("rain_mm"),
    col("hourly_data.snowfall").alias("snowfall_cm"),
    col("hourly_data.weathercode").alias("weather_code"),
    col("hourly_data.cloudcover").alias("cloud_cover_pct"),
    col("hourly_data.windspeed_10m").alias("windspeed_kmh"),
    col("hourly_data.windgusts_10m").alias("windgusts_kmh"),
    col("hourly_data.winddirection_10m").alias("wind_direction_deg"),
    col("hourly_data.surface_pressure").alias("surface_pressure_hpa")
)

# Deduplicate
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window \
    .partitionBy("city_name", "observation_time") \
    .orderBy(col("observation_time"))

df_flat = df_flat \
    .withColumn("rn", row_number().over(window_spec)) \
    .filter(col("rn") == 1) \
    .drop("rn")

print(f"Exploded rows after dedup: {df_flat.count()}")
df_flat.select("city_name").distinct().show()

Cities found:
+---------+
|city_name|
+---------+
|Singapore|
|  Toronto|
|    Dubai|
|   Sydney|
|   London|
|    Paris|
|   Berlin|
|   Mumbai|
|    Tokyo|
| New_York|
+---------+

Exploded rows after dedup: 240
+---------+
|city_name|
+---------+
|   Berlin|
|    Dubai|
|   London|
|   Mumbai|
| New_York|
|    Paris|
|Singapore|
|   Sydney|
|    Tokyo|
|  Toronto|
+---------+



In [0]:
df_silver = df_flat \
    .withColumn("observation_time",       to_timestamp(col("observation_time"), "yyyy-MM-dd'T'HH:mm")) \
    .withColumn("observation_date",       to_date(col("observation_time"))) \
    .withColumn("observation_hour",       hour(col("observation_time"))) \
    .withColumn("temperature_c",          col("temperature_c").cast(DoubleType())) \
    .withColumn("apparent_temperature_c", col("apparent_temperature_c").cast(DoubleType())) \
    .withColumn("relative_humidity_pct",  col("relative_humidity_pct").cast(IntegerType())) \
    .withColumn("dewpoint_c",             col("dewpoint_c").cast(DoubleType())) \
    .withColumn("precipitation_mm",       col("precipitation_mm").cast(DoubleType())) \
    .withColumn("rain_mm",                col("rain_mm").cast(DoubleType())) \
    .withColumn("snowfall_cm",            col("snowfall_cm").cast(DoubleType())) \
    .withColumn("weather_code",           col("weather_code").cast(IntegerType())) \
    .withColumn("cloud_cover_pct",        col("cloud_cover_pct").cast(IntegerType())) \
    .withColumn("windspeed_kmh",          col("windspeed_kmh").cast(DoubleType())) \
    .withColumn("windgusts_kmh",          col("windgusts_kmh").cast(DoubleType())) \
    .withColumn("wind_direction_deg",     col("wind_direction_deg").cast(IntegerType())) \
    .withColumn("surface_pressure_hpa",   col("surface_pressure_hpa").cast(DoubleType())) \
    .withColumn("ingestion_timestamp",    current_timestamp()) \
    .withColumn("ingestion_date",         current_date())

print(f"Silver rows: {df_silver.count()}")
df_silver.select("city_name", "observation_date").distinct().show(truncate=False)

Silver rows: 240
+---------+----------------+
|city_name|observation_date|
+---------+----------------+
|Berlin   |2024-01-02      |
|Dubai    |2024-01-02      |
|London   |2024-01-02      |
|Mumbai   |2024-01-02      |
|New_York |2024-01-02      |
|Paris    |2024-01-02      |
|Singapore|2024-01-02      |
|Sydney   |2024-01-02      |
|Tokyo    |2024-01-02      |
|Toronto  |2024-01-02      |
+---------+----------------+



In [0]:
dq_issues = []

# Check 1 — No nulls in critical columns
critical_cols = ["city_name", "observation_time", "temperature_c", "latitude", "longitude"]
for col_name in critical_cols:
    null_count = df_silver.filter(col(col_name).isNull()).count()
    if null_count > 0:
        dq_issues.append(f"NULL check FAILED: {col_name} has {null_count} nulls")
    else:
        print(f"✅ NULL check passed: {col_name}")

# Check 2 — Temperature in valid range
temp_outliers = df_silver.filter(
    (col("temperature_c") < -90) | (col("temperature_c") > 60)
).count()
if temp_outliers > 0:
    dq_issues.append(f"Temperature range FAILED: {temp_outliers} out-of-range values")
else:
    print(f"✅ Temperature range check passed")

# Check 3 — Humidity between 0 and 100
humidity_outliers = df_silver.filter(
    (col("relative_humidity_pct") < 0) | (col("relative_humidity_pct") > 100)
).count()
if humidity_outliers > 0:
    dq_issues.append(f"Humidity range FAILED: {humidity_outliers} out-of-range values")
else:
    print(f"✅ Humidity range check passed")

# Check 4 — FIXED: duplicate = same city + same observation_time
from pyspark.sql.functions import count as spark_count

duplicate_count = df_silver \
    .groupBy("city_name", "observation_time") \
    .agg(spark_count("*").alias("cnt")) \
    .filter(col("cnt") > 1) \
    .count()

if duplicate_count > 0:
    dq_issues.append(f"Duplicate check FAILED: {duplicate_count} city+time combos appear more than once")
else:
    print(f"✅ Duplicate check passed")

# Check 5 — All 10 cities present
expected_cities = {
    "New_York", "London", "Tokyo", "Dubai", "Mumbai",
    "Sydney", "Paris", "Berlin", "Toronto", "Singapore"
}

found_cities  = {r[0] for r in df_silver.select("city_name").distinct().collect()}
missing_cities = expected_cities - found_cities
city_count     = len(found_cities)

if missing_cities:
    dq_issues.append(
        f"City count FAILED: {city_count}/10 cities found. "
        f"Missing: {missing_cities}"
    )
else:
    print(f"✅ City count check passed: all 10 cities present")

# Check 6 — Expected rows: 10 cities x 24 hours x number of days in the file
actual_rows    = df_silver.count()
distinct_dates = df_silver.select("observation_date").distinct().count()
expected_rows  = 10 * 24 * distinct_dates

if actual_rows < expected_rows * 0.9:
    dq_issues.append(f"Row count FAILED: expected ~{expected_rows}, got {actual_rows}")
else:
    print(f"✅ Row count check passed: {actual_rows} rows across {distinct_dates} date(s)")

# Result
if dq_issues:
    for issue in dq_issues:
        print(f"❌ {issue}")
    raise Exception(f"DATA QUALITY FAILED: {len(dq_issues)} checks failed. Silver write aborted.")
else:
    print("\n✅ ALL DATA QUALITY CHECKS PASSED — proceeding to silver write")

✅ NULL check passed: city_name
✅ NULL check passed: observation_time
✅ NULL check passed: temperature_c
✅ NULL check passed: latitude
✅ NULL check passed: longitude
✅ Temperature range check passed
✅ Humidity range check passed
✅ Duplicate check passed
✅ City count check passed: all 10 cities present
✅ Row count check passed: 240 rows across 1 date(s)

✅ ALL DATA QUALITY CHECKS PASSED — proceeding to silver write


In [0]:
from delta.tables import DeltaTable

silver_table_name = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}"

# Use abfss path directly — NOT through volume
silver_table_path = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver/open_meteo/cleaned/"

table_exists = spark.catalog.tableExists(silver_table_name)

if not table_exists:
    print("First run — creating silver external Delta table...")
    
    # Write Delta files to ADLS first
    df_silver.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("observation_date", "city_name") \
        .save(silver_table_path)
    
    # Register as external table in Unity Catalog
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {silver_table_name}
        USING DELTA
        LOCATION '{silver_table_path}'
    """)
    
    print(f"✅ Silver external table created: {silver_table_name}")

else:
    print("Incremental run — merging into silver Delta table...")
    delta_table = DeltaTable.forName(spark, silver_table_name)

    delta_table.alias("target").merge(
        df_silver.alias("source"),
        "target.city_name = source.city_name AND target.observation_time = source.observation_time"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

    print(f"✅ Silver table merged: {silver_table_name}")

# Optimize
spark.sql(f"OPTIMIZE {silver_table_name} ZORDER BY (observation_hour, weather_code, temperature_c)")
print("✅ OPTIMIZE complete")

Incremental run — merging into silver Delta table...
✅ Silver table merged: weather_catalog.silver.weather_hourly_cleaned
✅ OPTIMIZE complete


In [0]:
%sql
DROP VOLUME IF EXISTS weather_catalog.bronze.raw_volume;
DROP VOLUME IF EXISTS weather_catalog.silver.cleaned_volume;
DROP VOLUME IF EXISTS weather_catalog.gold.star_schema_volume;

In [0]:
from delta.tables import DeltaTable

silver_table_name = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}"
silver_table_path = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver/open_meteo/cleaned/"

table_exists = spark.catalog.tableExists(silver_table_name)

if not table_exists:
    print("First run — writing Delta files and registering external table...")

    # Step A — Write Delta files directly to ADLS
    df_silver.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("observation_date", "city_name") \
        .save(silver_table_path)
    
    print(f"✅ Delta files written to: {silver_table_path}")

    # Step B — Register as external table pointing to that path
    spark.sql(f"""
        CREATE TABLE {silver_table_name}
        USING DELTA
        LOCATION '{silver_table_path}'
    """)
    
    print(f"✅ External table registered: {silver_table_name}")

else:
    print("Incremental run — merging into silver Delta table...")
    
    delta_table = DeltaTable.forName(spark, silver_table_name)

    delta_table.alias("target").merge(
        df_silver.alias("source"),
        """target.city_name = source.city_name 
           AND target.observation_time = source.observation_time"""
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

    print(f"✅ Silver table merged successfully")

# Optimize for query performance
spark.sql(f"OPTIMIZE {silver_table_name} ZORDER BY (observation_hour, weather_code, temperature_c)")
print("✅ OPTIMIZE with ZORDER complete")

Incremental run — merging into silver Delta table...
✅ Silver table merged successfully
✅ OPTIMIZE with ZORDER complete


In [0]:
# Final verification
df_verify = spark.table(silver_table_name)
print(f"Total rows in silver: {df_verify.count()}")
print(f"Date range: {df_verify.agg(min('observation_date'), max('observation_date')).collect()}")
print(f"Cities: {[r[0] for r in df_verify.select('city_name').distinct().collect()]}")

df_verify.show(5)

Total rows in silver: 240
Date range: [Row(min(observation_date)=datetime.date(2024, 1, 2), max(observation_date)=datetime.date(2024, 1, 2))]
Cities: ['Singapore', 'Mumbai', 'Paris', 'Sydney', 'New_York', 'Berlin', 'Toronto', 'Tokyo', 'Dubai', 'London']
+---------+---------+---------+---------+--------------+---------------------+------------------+-------------------+-------------+----------------------+---------------------+----------+----------------+-------+-----------+------------+---------------+-------------+-------------+------------------+--------------------+----------------+----------------+--------------------+--------------+
|city_name| latitude|longitude|elevation|      timezone|timezone_abbreviation|utc_offset_seconds|   observation_time|temperature_c|apparent_temperature_c|relative_humidity_pct|dewpoint_c|precipitation_mm|rain_mm|snowfall_cm|weather_code|cloud_cover_pct|windspeed_kmh|windgusts_kmh|wind_direction_deg|surface_pressure_hpa|observation_date|observation_hour

In [0]:
# Count how many JSON files exist across all of bronze
import subprocess
all_files = dbutils.fs.ls('abfss://weather-data@weatherdatalake.dfs.core.windows.net/bronze/open_meteo/raw/')
print(f"Top level folders: {len(all_files)}")
for f in all_files:
    print(f.path)

Top level folders: 1
abfss://weather-data@weatherdatalake.dfs.core.windows.net/bronze/open_meteo/raw/2026/


timepass


In [0]:
from datetime import datetime, timezone

def get_widget(name, default):
    try:
        dbutils.widgets.remove(name)
    except:
        pass
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name)

start_date = get_widget("p_start_date", "2024-01-01")
end_date   = get_widget("p_end_date",   "2024-01-01")

# Always compute today fresh — never rely on widget for this
ingestion_date = datetime.now(timezone.utc).strftime("%Y/%m/%d")

print(f"start_date:     {start_date}")
print(f"end_date:       {end_date}")
print(f"ingestion_date: {ingestion_date}")

start_date:     
end_date:       
ingestion_date: 2026/05/24


In [0]:
from pyspark.sql.functions import col, regexp_extract

# Build path from freshly computed ingestion_date — never from widget
bronze_path = (
    f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}"
    f".dfs.core.windows.net/bronze/open_meteo/raw/{ingestion_date}/"
)

print(f"Reading: {bronze_path}")

# Verify path exists
try:
    city_folders = dbutils.fs.ls(bronze_path)
    print(f"Found {len(city_folders)} folders: {[f.name for f in city_folders]}")
except Exception as e:
    raise Exception(f"Bronze path not found: {bronze_path} | Error: {e}")

# Read all city files
df_raw = spark.read \
    .option("multiline", "true") \
    .option("recursiveFileLookup", "true") \
    .json(bronze_path)

print(f"Raw records: {df_raw.count()}")

Reading: abfss://weather-data@weatherdatalake.dfs.core.windows.net/bronze/open_meteo/raw/2026/05/24/
Found 10 folders: ['Berlin/', 'Dubai/', 'London/', 'Mumbai/', 'New_York/', 'Paris/', 'Singapore/', 'Sydney/', 'Tokyo/', 'Toronto/']
Raw records: 10


In [0]:
from pyspark.sql.functions import arrays_zip, explode

df_with_meta = df_raw.withColumn("source_file", col("_metadata.file_path"))

# Extract city from second-to-last path segment
# .../2026/05/24/Singapore/Singapore_2024-01-02.json → Singapore
df_with_meta = df_with_meta.withColumn(
    "city_name",
    regexp_extract(col("source_file"), r"/([^/]+)/[^/]+\.json$", 1)
)

print("Cities found:")
df_with_meta.select("city_name").distinct().show()

# Zip hourly arrays
df_zipped = df_with_meta.select(
    col("city_name"),
    col("latitude"),
    col("longitude"),
    col("elevation"),
    col("timezone"),
    col("timezone_abbreviation"),
    col("utc_offset_seconds"),
    arrays_zip(
        col("hourly.time"),
        col("hourly.temperature_2m"),
        col("hourly.apparent_temperature"),
        col("hourly.relativehumidity_2m"),
        col("hourly.dewpoint_2m"),
        col("hourly.precipitation"),
        col("hourly.rain"),
        col("hourly.snowfall"),
        col("hourly.weathercode"),
        col("hourly.cloudcover"),
        col("hourly.windspeed_10m"),
        col("hourly.windgusts_10m"),
        col("hourly.winddirection_10m"),
        col("hourly.surface_pressure")
    ).alias("hourly_zipped")
)

df_exploded = df_zipped.withColumn("hourly_data", explode(col("hourly_zipped")))

df_flat = df_exploded.select(
    col("city_name"),
    col("latitude"),
    col("longitude"),
    col("elevation"),
    col("timezone"),
    col("timezone_abbreviation"),
    col("utc_offset_seconds"),
    col("hourly_data.time").alias("observation_time"),
    col("hourly_data.temperature_2m").alias("temperature_c"),
    col("hourly_data.apparent_temperature").alias("apparent_temperature_c"),
    col("hourly_data.relativehumidity_2m").alias("relative_humidity_pct"),
    col("hourly_data.dewpoint_2m").alias("dewpoint_c"),
    col("hourly_data.precipitation").alias("precipitation_mm"),
    col("hourly_data.rain").alias("rain_mm"),
    col("hourly_data.snowfall").alias("snowfall_cm"),
    col("hourly_data.weathercode").alias("weather_code"),
    col("hourly_data.cloudcover").alias("cloud_cover_pct"),
    col("hourly_data.windspeed_10m").alias("windspeed_kmh"),
    col("hourly_data.windgusts_10m").alias("windgusts_kmh"),
    col("hourly_data.winddirection_10m").alias("wind_direction_deg"),
    col("hourly_data.surface_pressure").alias("surface_pressure_hpa")
)

# Deduplicate
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window \
    .partitionBy("city_name", "observation_time") \
    .orderBy(col("observation_time"))

df_flat = df_flat \
    .withColumn("rn", row_number().over(window_spec)) \
    .filter(col("rn") == 1) \
    .drop("rn")

print(f"Exploded rows after dedup: {df_flat.count()}")
df_flat.select("city_name").distinct().show()

Cities found:
+---------+
|city_name|
+---------+
|Singapore|
|  Toronto|
|    Dubai|
|   Sydney|
|   London|
|    Paris|
|   Berlin|
|   Mumbai|
|    Tokyo|
| New_York|
+---------+

Exploded rows after dedup: 240
+---------+
|city_name|
+---------+
|   Berlin|
|    Dubai|
|   London|
|   Mumbai|
| New_York|
|    Paris|
|Singapore|
|   Sydney|
|    Tokyo|
|  Toronto|
+---------+

